## 4. Data loading, canonicalisation and curation

The structures are those of the Therapeutics Data Commons Ames benchmark, a curated
compilation of 7,255 compounds derived from the Hansen et al. dataset. The SMILES strings
and mutagenicity labels are read from three CSV files placed in the working directory and
concatenated into a single pool; the partitioning applied to that pool is defined in the
next section and does not depend on how the files were grouped.

Structures are converted to canonical SMILES with RDKit. Compounds that fail to parse are
discarded, as are compounds whose duplicate entries carry contradictory labels; remaining
duplicates are collapsed to a single record. Each count is recorded so that the curation
is auditable and so that a change in the source compilation would be visible immediately.

## 1. Environment and persistent storage

In [1]:
!pip install -q rdkit
!pip install -q xgboost
!pip install -q optuna
!pip install -q scikit-posthocs
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.3 MB/s eta 0:00:00


In [2]:
# All artefacts are written to Google Drive so that long searches survive
# a runtime disconnection and can resume from the last completed model.
from google.colab import drive
drive.mount('/content/drive')

import os
WORK    = "/content/drive/MyDrive/ames_pipeline"
DATADIR = f"{WORK}/data"          # source SMILES tables
OUTDIR  = f"{WORK}/results"
CKPT    = f"{WORK}/ckpt"
FIGDIR  = f"{WORK}/figures"
for d in (WORK, DATADIR, OUTDIR, CKPT, FIGDIR):
    os.makedirs(d, exist_ok=True)
print("Working directory:", WORK)
print("Place train.csv, valid.csv and test.csv in:", DATADIR)

Mounted at /content/drive
Working directory: /content/drive/MyDrive/ames_pipeline
Place train.csv, valid.csv and test.csv in: /content/drive/MyDrive/ames_pipeline/data


## 2. Configuration

All quantities that govern the experimental design are collected here.

The primary split is drawn with a seed disjoint from the multi-split benchmark, so that
the analyses carried out on a single designated model are independent of the splits used
for the family-level comparison. Fifteen scaffold partitions give a Nemenyi critical
difference of 1.58 rank positions for five models; an equal number of random partitions
allows the cost of the easier partitioning scheme to be estimated with matched precision.

The search and the final fits share a single training protocol, so that the budget a
configuration receives during selection equals the budget it receives afterwards.

In [8]:
# ---------------------------- CONFIGURATION ----------------------------
SEED = 42

# Partitioning
PRIMARY_SPLIT_SEED   = 101                    # designated split, disjoint from the benchmark
SCAFFOLD_SPLIT_SEEDS = list(range(42, 57))   # 15 scaffold partitions
RANDOM_SPLIT_SEEDS   = list(range(42, 57))   # 15 random partitions of identical proportions

# Hyperparameter search: identical trial budget for every model
N_TRIALS = 100

# Training protocol, shared by the search and the final fits
MAX_EPOCHS = 250
PATIENCE   = 30

# Seed ensembling, applied identically to both families
SEEDS = [42, 43, 44]

# Molecular representations
FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
# -----------------------------------------------------------------------
print(f"{len(SCAFFOLD_SPLIT_SEEDS)} scaffold and {len(RANDOM_SPLIT_SEEDS)} random partitions")
print(f"{N_TRIALS} trials per model, up to {MAX_EPOCHS} epochs, patience {PATIENCE}")

15 scaffold and 15 random partitions
100 trials per model, up to 250 epochs, patience 30


## 3. Common utilities

In [4]:
import os, random, json, time, warnings
import numpy as np, pandas as pd, torch
from collections import defaultdict
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

def set_seed(s):
    """Fix the random state of Python, NumPy and PyTorch."""
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(SEED)

def save_ckpt(name, obj):
    with open(f"{CKPT}/{name}.json", "w") as f:
        json.dump(obj, f, indent=2)

def load_ckpt(name):
    path = f"{CKPT}/{name}.json"
    return json.load(open(path)) if os.path.exists(path) else None

Device: cuda | Tesla T4


## 4. Data loading, canonicalisation and curation

Structures are read from the Therapeutics Data Commons Ames benchmark and converted to
canonical SMILES with RDKit. Compounds that fail to parse are discarded, as are compounds
whose duplicate entries carry contradictory labels; the remaining duplicates are collapsed
to a single record. Each of the three counts is recorded so that the curation is auditable.

In [6]:
SOURCE_FILES = ["train.csv", "valid.csv", "test.csv"]
EXPECTED_N   = 7255

missing = [f for f in SOURCE_FILES if not os.path.exists(f"{DATADIR}/{f}")]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {DATADIR}. "
        "Upload the source SMILES tables to that folder before running this cell.")

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in SOURCE_FILES],
               ignore_index=True)[["Drug_ID", "smiles", "label"]]
n_raw = len(df)
if n_raw != EXPECTED_N:
    print(f"Note: {n_raw} compounds read, {EXPECTED_N} expected for this benchmark version.")

def canonical_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(mol) if mol is not None else None

df["canonical"] = df["smiles"].apply(canonical_smiles)
df = df.dropna(subset=["canonical"]).copy()
n_invalid = n_raw - len(df)

conflicting = (df.groupby("canonical")["label"].nunique()
                 .reset_index().query("label > 1")["canonical"].tolist())
df = df[~df["canonical"].isin(conflicting)].copy()

n_before_dedup = len(df)
df = df.drop_duplicates(subset="canonical", keep="first").reset_index(drop=True)
n_duplicate = n_before_dedup - len(df)

n_pos = int(df["label"].sum())
summary = {
    "n_raw":               n_raw,
    "n_invalid_dropped":   n_invalid,
    "n_conflict_dropped":  len(conflicting),
    "n_duplicate_dropped": n_duplicate,
    "n_total":             len(df),
    "n_positive":          n_pos,
    "n_negative":          len(df) - n_pos,
    "positive_ratio":      round(n_pos / len(df), 4),
}
pd.DataFrame(summary.items(), columns=["metric", "value"]).to_csv(
    f"{OUTDIR}/data_summary.csv", index=False)

for k, v in summary.items():
    print(f"  {k:22s}: {v}")

  n_raw                 : 7255
  n_invalid_dropped     : 0
  n_conflict_dropped    : 0
  n_duplicate_dropped   : 0
  n_total               : 7255
  n_positive            : 3951
  n_negative            : 3304
  positive_ratio        : 0.5446


## 5. Scaffold and random partitioning

Bemis–Murcko scaffolds are assigned to a single partition, so that no scaffold present in
the training set appears in the test set. Scaffold groups are ordered by descending size
with random tie-breaking and filled greedily into an 80/10/10 partition.

A random partition of identical proportions is defined for the same data, and is used
later to quantify how much of the reported performance depends on the partitioning scheme
rather than on the model.

The class balance of each partition is reported: scaffold partitioning does not preserve
the overall positive rate, and the resulting shift is relevant to the calibration and
conformal analyses.

In [9]:
def safe_scaffold(smi):
    """Bemis-Murcko scaffold, falling back to the whole molecule when undefined."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try:
            return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception:
            return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs  = list(groups.values())
    rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d)
    n_train, n_train_valid = int(frac[0] * n), int((frac[0] + frac[1]) * n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr) + len(g) <= n_train:
            tr += g
        elif len(tr) + len(va) + len(g) <= n_train_valid:
            va += g
        else:
            te += g
    return d.loc[tr], d.loc[va], d.loc[te]

def make_random_split(d, seed, frac=(0.8, 0.1, 0.1)):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(d.index.values)
    n = len(d)
    a, b = int(frac[0] * n), int((frac[0] + frac[1]) * n)
    return d.loc[idx[:a]], d.loc[idx[a:b]], d.loc[idx[b:]]

df_all = df.copy()
df_train, df_valid, df_test = [x.reset_index(drop=True)
    for x in make_scaffold_split(df_all, seed=PRIMARY_SPLIT_SEED)]

for name, d in [("Train", df_train), ("Valid", df_valid), ("Test", df_test)]:
    print(f"{name}: n = {len(d):5d}, positive = {100 * d['label'].mean():.1f}%")

train_scaffolds = {safe_scaffold(s) for s in df_train["canonical"]}
test_scaffolds  = {safe_scaffold(s) for s in df_test["canonical"]}
leak = 100 * len(test_scaffolds & train_scaffolds) / max(len(test_scaffolds), 1)
print(f"Test scaffolds also present in training: {leak:.1f}%")

assert PRIMARY_SPLIT_SEED not in SCAFFOLD_SPLIT_SEEDS, \
    "The primary split must be disjoint from the multi-split benchmark."

for name, d in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    d.to_csv(f"{OUTDIR}/{name}.csv", index=False)

Train: n =  5804, positive = 55.6%
Valid: n =   725, positive = 53.7%
Test: n =   726, positive = 46.3%
Test scaffolds also present in training: 0.0%


## 6. Circular fingerprints

The classical models operate on 2048-bit Morgan fingerprints of radius 2, which encode
each atom environment together with the bond types it contains.

In [10]:
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr

def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None:
            X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)

X_train, df_train = build_X(df_train); y_train = df_train["label"].values
X_valid, df_valid = build_X(df_valid); y_valid = df_valid["label"].values
X_test,  df_test  = build_X(df_test);  y_test  = df_test["label"].values
print("Fingerprint matrices:", X_train.shape, X_valid.shape, X_test.shape)

Fingerprint matrices: (5804, 2048) (725, 2048) (726, 2048)


## 7. Molecular graphs

Each molecule is represented as an undirected graph. Atoms carry a 24-dimensional feature
vector: one-hot element and hybridisation indicators together with degree, formal charge,
aromaticity, hydrogen count and ring membership. Bonds carry a 6-dimensional vector
recording bond order, aromaticity, conjugation and ring membership.

Bond features are supplied to the message-passing layers, so that the graph models have
access to the same bond information that the circular fingerprints encode.

In [11]:
from torch_geometric.data import Data

ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']

def one_hot(x, allowable):
    if x not in allowable:
        x = allowable[-1]
    return [int(x == a) for a in allowable]

def atom_features(atom):
    f  = one_hot(atom.GetSymbol(), ATOM_LIST)
    f += one_hot(str(atom.GetHybridization()), HYBRIDIZATION)
    f += [atom.GetDegree(), atom.GetFormalCharge(), int(atom.GetIsAromatic()),
          atom.GetTotalNumHs(), int(atom.IsInRing())]
    return f

def bond_features(bond):
    bt = bond.GetBondType()
    return [int(bt == Chem.BondType.SINGLE),
            int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE),
            int(bt == Chem.BondType.AROMATIC),
            int(bond.GetIsConjugated()),
            int(bond.IsInRing())]

def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_index_list, edge_attr_list = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index_list += [[i, j], [j, i]]
        edge_attr_list  += [bf, bf]
    if len(edge_index_list) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(edge_attr_list, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)

def build_dataset(d):
    graphs = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [g for g in graphs if g is not None]

train_set = build_dataset(df_train)
valid_set = build_dataset(df_valid)
test_set  = build_dataset(df_test)
print(f"Graphs: train = {len(train_set)}, valid = {len(valid_set)}, test = {len(test_set)}")

Graphs: train = 5804, valid = 725, test = 726


## 8. Model architectures

Three message-passing architectures are used. GINE and GAT consume bond features
directly; GCN operates on connectivity alone, since the graph convolution of Kipf and
Welling admits scalar edge weights but not edge feature vectors. This makes GCN a useful
reference point for the contribution of bond information.

Node features are projected to the hidden dimension before the first message-passing
layer, so that every layer maps the hidden dimension onto itself. Residual connections
are treated as a hyperparameter rather than fixed in advance, since the appropriate depth
regularisation is not known beforehand.

In [12]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import (GCNConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)

class GNN(nn.Module):
    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn":
                conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp  = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                     nn.Linear(hidden, hidden))
                conv = GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads, edge_dim=edge_dim)
            else:
                raise ValueError(f"Unknown architecture: {kind}")
            self.convs.append(conv)
            self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))

    def forward(self, data):
        x, edge_index, edge_attr, batch = (data.x, data.edge_index,
                                           data.edge_attr, data.batch)
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(x, edge_index) if self.kind == "gcn" else conv(x, edge_index, edge_attr)
            h = F.relu(bn(h))
            h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, batch), global_add_pool(x, batch)], dim=1)
        return self.head(g).squeeze(-1)

def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads).to(DEVICE)

ARCHS = ["gcn", "gin", "gat"]

from torch_geometric.loader import DataLoader
probe = next(iter(DataLoader(train_set[:8], batch_size=8))).to(DEVICE)
for kind in ARCHS:
    model = build_gnn(kind, 64, 2, 0.1)
    model.eval()
    with torch.no_grad():
        out = model(probe)
    n_par = sum(p.numel() for p in model.parameters())
    print(f"  {kind.upper():4s} output {tuple(out.shape)}, {n_par:,} parameters")

  GCN  output (8,), 18,497 parameters
  GIN  output (8,), 27,715 parameters
  GAT  output (8,), 19,649 parameters


## 9. Training protocol

A single routine trains every graph model, and it is used unchanged during the
hyperparameter search, for the final fits and for the multi-split benchmark. Optimisation
uses Adam with a class-weighted binary cross-entropy loss; the epoch with the highest
validation ROC-AUC is retained and training stops after `PATIENCE` epochs without
improvement.

When an Optuna trial is supplied, the validation score is reported after each epoch so
that clearly unpromising configurations can be terminated early. Pruning removes weak
configurations; it does not cap the budget available to the selected one.

In [13]:
from sklearn.metrics import roc_auc_score

pos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)],
                          dtype=torch.float, device=DEVICE)

@torch.no_grad()
def evaluate(model, loader):
    """Return labels, predicted probabilities and logits for a data loader."""
    model.eval()
    ys, ps, logits = [], [], []
    for d in loader:
        d = d.to(DEVICE)
        logit = model(d)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(logit).cpu().numpy())
        logits.append(logit.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(logits)

def train_gnn(kind, params, seed, tr_set, va_set,
              max_epochs=MAX_EPOCHS, patience=PATIENCE, trial=None, pos_w=None):
    """Train one graph model and return it together with its best validation ROC-AUC."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"],
                      params["dropout"], params.get("residual", True))
    optimiser = torch.optim.Adam(model.parameters(), lr=params["lr"],
                                 weight_decay=params["weight_decay"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w if pos_w is not None else pos_weight)
    train_loader = DataLoader(tr_set, batch_size=params["batch_size"],
                              shuffle=True, drop_last=True)
    valid_loader = DataLoader(va_set, batch_size=256)

    best_auc, best_state, waited = -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        for d in train_loader:
            d = d.to(DEVICE)
            optimiser.zero_grad()
            criterion(model(d), d.y.view(-1)).backward()
            optimiser.step()

        y_va, p_va, _ = evaluate(model, valid_loader)
        auc = roc_auc_score(y_va, p_va) if len(np.unique(y_va)) > 1 else 0.5

        if trial is not None:
            trial.report(auc, epoch)
            if trial.should_prune():
                import optuna
                raise optuna.TrialPruned()

        if auc > best_auc:
            best_auc, waited = auc, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc

## 10. Hyperparameter search

Every model receives the same number of trials. Both families use a TPE sampler with a
median pruner, and the wall-clock time consumed by each search is recorded, since an equal
number of trials does not imply an equal amount of computation.

Results are checkpointed after each model, so an interrupted session resumes from the last
completed search.

In [14]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

def objective_xgb(trial):
    model = XGBClassifier(
        n_estimators=1500,
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 12),
        scale_pos_weight=scale_pos, eval_metric="auc", early_stopping_rounds=50,
        tree_method="hist", random_state=SEED, n_jobs=-1)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    return roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])

def objective_rf(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 200, 1200, step=100),
        max_depth=trial.suggest_categorical("max_depth", [None, 10, 20, 30, 50]),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 8),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 12),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", 0.1, 0.3]),
        class_weight="balanced", random_state=SEED, n_jobs=-1)
    model.fit(X_train, y_train)
    return roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])

best_classic = load_ckpt("hpo_classic") or {}
for name, objective in [("XGBoost", objective_xgb), ("RandomForest", objective_rf)]:
    if name in best_classic:
        print(f"{name}: loaded from checkpoint")
        continue
    t0 = time.time()
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    elapsed = time.time() - t0
    best_classic[name] = {"params": study.best_params, "valid_auc": study.best_value,
                          "best_trial": study.best_trial.number, "seconds": elapsed}
    save_ckpt("hpo_classic", best_classic)
    print(f"{name:14s} validation ROC-AUC = {study.best_value:.4f}  "
          f"(trial {study.best_trial.number}, {elapsed / 60:.1f} min)")
    print(f"   {study.best_params}")

  0%|          | 0/100 [00:00<?, ?it/s]

XGBoost        validation ROC-AUC = 0.8389  (trial 90, 2.2 min)
   {'max_depth': 11, 'learning_rate': 0.15642823651322757, 'subsample': 0.6396975505413772, 'colsample_bytree': 0.5821188098787602, 'reg_lambda': 0.06842025683573061, 'reg_alpha': 0.03908769860097184, 'min_child_weight': 4}


  0%|          | 0/100 [00:00<?, ?it/s]

RandomForest   validation ROC-AUC = 0.8335  (trial 72, 15.3 min)
   {'n_estimators': 900, 'max_depth': 30, 'min_samples_leaf': 1, 'min_samples_split': 4, 'max_features': 'sqrt'}


In [15]:
def make_objective(kind):
    def objective(trial):
        params = {
            "hidden":       trial.suggest_categorical("hidden", [64, 128, 256]),
            "n_layers":     trial.suggest_int("n_layers", 2, 5),
            "dropout":      trial.suggest_float("dropout", 0.0, 0.5),
            "lr":           trial.suggest_float("lr", 1e-4, 5e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
            "batch_size":   trial.suggest_categorical("batch_size", [32, 64, 128]),
            "residual":     trial.suggest_categorical("residual", [True, False]),
        }
        _, auc = train_gnn(kind, params, SEED, train_set, valid_set, trial=trial)
        return auc
    return objective

best_gnn = load_ckpt("hpo_gnn") or {}
for kind in ARCHS:
    if kind in best_gnn:
        print(f"{kind.upper()}: loaded from checkpoint")
        continue
    t0 = time.time()
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
    study.optimize(make_objective(kind), n_trials=N_TRIALS, show_progress_bar=True)
    elapsed = time.time() - t0
    n_pruned = len([t for t in study.trials
                    if t.state == optuna.trial.TrialState.PRUNED])
    best_gnn[kind] = {"params": study.best_params, "valid_auc": study.best_value,
                      "best_trial": study.best_trial.number, "seconds": elapsed,
                      "n_pruned": n_pruned}
    save_ckpt("hpo_gnn", best_gnn)
    print(f"{kind.upper():5s} validation ROC-AUC = {study.best_value:.4f}  "
          f"(trial {study.best_trial.number}, {elapsed / 60:.1f} min, {n_pruned} pruned)")
    print(f"   {study.best_params}")

  0%|          | 0/100 [00:00<?, ?it/s]

GCN   validation ROC-AUC = 0.8493  (trial 72, 63.1 min, 39 pruned)
   {'hidden': 256, 'n_layers': 3, 'dropout': 0.05358601947429411, 'lr': 0.004132189266590333, 'weight_decay': 4.705119022984706e-06, 'batch_size': 128, 'residual': True}


  0%|          | 0/100 [00:00<?, ?it/s]

GIN   validation ROC-AUC = 0.8519  (trial 85, 69.3 min, 56 pruned)
   {'hidden': 64, 'n_layers': 4, 'dropout': 0.20731860936937893, 'lr': 0.0018017848240952697, 'weight_decay': 2.6566273102091933e-06, 'batch_size': 32, 'residual': False}


  0%|          | 0/100 [00:00<?, ?it/s]

GAT   validation ROC-AUC = 0.8466  (trial 75, 104.8 min, 52 pruned)
   {'hidden': 128, 'n_layers': 3, 'dropout': 0.11678477971374823, 'lr': 0.00093642486119946, 'weight_decay': 0.0005566438039612761, 'batch_size': 32, 'residual': True}


## 11. Selected configurations

The search outcome is written to `hpo_budget.csv`, which records the selected
configuration, the validation score that motivated it, the trial at which it was found and
the computation the search consumed for each model.

In [16]:
rows = []
for family, store, label in [("classical", best_classic, lambda k: k),
                             ("graph",     best_gnn,     lambda k: k.upper())]:
    for key, value in store.items():
        rows.append({
            "family":          family,
            "model":           label(key),
            "n_trials":        N_TRIALS,
            "best_valid_auc":  value["valid_auc"],
            "best_trial":      value["best_trial"],
            "wall_clock_min":  value["seconds"] / 60,
            "n_pruned":        value.get("n_pruned", 0),
            "params":          json.dumps(value["params"]),
        })

hpo_df = pd.DataFrame(rows)
hpo_df.to_csv(f"{OUTDIR}/hpo_budget.csv", index=False)

json.dump({k: v["params"] for k, v in best_classic.items()},
          open(f"{OUTDIR}/hpo_classic.json", "w"), indent=2)
json.dump({k: v["params"] for k, v in best_gnn.items()},
          open(f"{OUTDIR}/hpo_gnn.json", "w"), indent=2)

print(hpo_df[["family", "model", "best_valid_auc", "best_trial",
              "wall_clock_min", "n_pruned"]].round(4).to_string(index=False))
print()
for _, r in hpo_df.iterrows():
    print(f"  {r.model:14s} {r.params}")

   family        model  best_valid_auc  best_trial  wall_clock_min  n_pruned
classical      XGBoost          0.8389          90          2.1855         0
classical RandomForest          0.8335          72         15.2593         0
    graph          GCN          0.8493          72         63.1395        39
    graph          GIN          0.8519          85         69.2882        56
    graph          GAT          0.8466          75        104.7790        52

  XGBoost        {"max_depth": 11, "learning_rate": 0.15642823651322757, "subsample": 0.6396975505413772, "colsample_bytree": 0.5821188098787602, "reg_lambda": 0.06842025683573061, "reg_alpha": 0.03908769860097184, "min_child_weight": 4}
  RandomForest   {"n_estimators": 900, "max_depth": 30, "min_samples_leaf": 1, "min_samples_split": 4, "max_features": "sqrt"}
  GCN            {"hidden": 256, "n_layers": 3, "dropout": 0.05358601947429411, "lr": 0.004132189266590333, "weight_decay": 4.705119022984706e-06, "batch_size": 128, "resid